# GCN vs GAT Training — Metrica Pass Graphs

**Task:** Given a snapshot of all 22 players at the moment a pass occurs,  
predict which team is making the pass (Home=0, Away=1).

This tests whether the GNN can learn team-specific spatial patterns —  
formation shape, pressing structure, and field position — purely from graph topology.

**Dataset:** 799 graphs from Metrica Sample Game 1 (437 Home, 362 Away)

**Models:** FootballGCN vs FootballGAT (with edge features)

**Sections**
1. Load & relabel dataset
2. Train/val/test split + DataLoader
3. Training loop
4. GCN experiment
5. GAT experiment
6. Head-to-head comparison

In [ ]:
import sys, os

# Set repo root explicitly for this project
REPO_ROOT = "/Users/eric/projects/football-analysis"

if not os.path.exists(os.path.join(REPO_ROOT, "src")):
    raise RuntimeError(f"Repository not found at {REPO_ROOT}")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for script execution
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

from src.models.gcn import FootballGCN
from src.models.gat import FootballGAT

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cpu")
print("Repo root:", REPO_ROOT)
print("Device:   ", DEVICE)

## 1. Load & Relabel Dataset

In [ ]:
PROCESSED  = os.path.join(REPO_ROOT, "data", "processed", "metrica_game1_pass_graphs.pt")
EVENTS_CSV = os.path.join(REPO_ROOT, "data", "Sample_Game_1", "Sample_Game_1_RawEventsData.csv")

raw_graphs = torch.load(PROCESSED, weights_only=False)

events = pd.read_csv(EVENTS_CSV)
events.columns = [
    c.strip().lower().replace(" ", "_").replace("[", "").replace("]", "")
    for c in events.columns
]
pass_events = events[events["type"] == "PASS"].reset_index(drop=True)
events      = events.reset_index(drop=True)

assert len(raw_graphs) == len(pass_events), "Graph/event count mismatch"

dataset: list[Data] = []
for g, (_, row) in zip(raw_graphs, pass_events.iterrows()):
    g = g.clone()
    team_label = 0.0 if row["team"] == "Home" else 1.0

    sx, ex = row.get("start_x", float("nan")), row.get("end_x", float("nan"))
    if not (pd.isna(sx) or pd.isna(ex)):
        forward = 1.0 if (row["team"] == "Home" and ex > sx) or \
                         (row["team"] == "Away" and ex < sx) else 0.0
    else:
        forward = float("nan")

    g.y     = torch.tensor([team_label], dtype=torch.float)
    g.y_fwd = torch.tensor([forward],    dtype=torch.float)
    # Drop team flag (col 4): [x, y, vx, vy, dist_atk, dist_def, angle, pressure]
    # Without it the model must learn from spatial structure alone
    g.x = torch.cat([g.x[:, :4], g.x[:, 5:]], dim=1)
    dataset.append(g)

labels = [int(g.y.item()) for g in dataset]
print(f"Dataset: {len(dataset)} graphs")
print(f"  Home passes (0): {labels.count(0)}")
print(f"  Away passes (1): {labels.count(1)}")
print(f"  Node feat dim  : {dataset[0].x.shape[1]}  (team flag removed)")
print(f"  Edge feat dim  : {dataset[0].edge_attr.shape[1]}")

## 2. Train / Val / Test Split

In [ ]:
from torch_geometric.loader import DataLoader

# Chronological split — no shuffling, respects temporal order
n      = len(dataset)
n_tr   = int(0.70 * n)
n_val  = int(0.15 * n)

train_data = dataset[:n_tr]
val_data   = dataset[n_tr : n_tr + n_val]
test_data  = dataset[n_tr + n_val :]

BATCH = 32
train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH)
test_loader  = DataLoader(test_data,  batch_size=BATCH)

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

## 3. Training & Evaluation Utilities

In [ ]:
def train_epoch(model, loader, optimizer, use_edge_attr=False):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        ea = batch.edge_attr if use_edge_attr else None
        if use_edge_attr:
            out = model(batch.x, batch.edge_index, batch.batch, edge_attr=ea)
        else:
            out = model(batch.x, batch.edge_index, batch.batch)

        loss = F.binary_cross_entropy_with_logits(out.squeeze(), batch.y.squeeze())
        loss.backward()
        optimizer.step()

        total_loss += float(loss) * batch.num_graphs
        preds = (out.squeeze() > 0).long()
        correct += int((preds == batch.y.squeeze().long()).sum())
        total += batch.num_graphs
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, use_edge_attr=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        ea = batch.edge_attr if use_edge_attr else None
        if use_edge_attr:
            out = model(batch.x, batch.edge_index, batch.batch, edge_attr=ea)
        else:
            out = model(batch.x, batch.edge_index, batch.batch)

        loss = F.binary_cross_entropy_with_logits(out.squeeze(), batch.y.squeeze())
        total_loss += float(loss) * batch.num_graphs
        preds = (out.squeeze() > 0).long()
        correct += int((preds == batch.y.squeeze().long()).sum())
        total += batch.num_graphs
        all_probs.extend(torch.sigmoid(out.squeeze()).tolist())
        all_labels.extend(batch.y.squeeze().long().tolist())

    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / total, correct / total, auc, all_probs, all_labels


def run_training(model, optimizer, n_epochs, use_edge_attr=False, scheduler=None):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_auc": []}
    best_val_auc, best_state = 0.0, None

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, use_edge_attr)
        va_loss, va_acc, va_auc, _, _ = evaluate(model, val_loader, use_edge_attr)

        if scheduler:
            scheduler.step(va_loss)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        history["val_auc"].append(va_auc)

        if va_auc > best_val_auc:
            best_val_auc = va_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}  "
                  f"train loss={tr_loss:.4f} acc={tr_acc:.3f}  "
                  f"val loss={va_loss:.4f} acc={va_acc:.3f} auc={va_auc:.3f}")

    model.load_state_dict(best_state)
    return history


def plot_training(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], label="Train", color="#1565c0")
    axes[0].plot(epochs, history["val_loss"],   label="Val",   color="#b71c1c")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

    axes[1].plot(epochs, history["train_acc"], label="Train", color="#1565c0")
    axes[1].plot(epochs, history["val_acc"],   label="Val",   color="#b71c1c")
    axes[1].set_ylim(0, 1); axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")

    axes[2].plot(epochs, history["val_auc"], color="#2e7d32")
    axes[2].set_ylim(0, 1); axes[2].set_title("Val ROC-AUC"); axes[2].set_xlabel("Epoch")

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()
    return fig

## 4. GCN Experiment

In [ ]:
IN_CH    = dataset[0].x.shape[1]       # 9
EDGE_DIM = dataset[0].edge_attr.shape[1]  # 6
EPOCHS   = 80

gcn = FootballGCN(
    in_channels=IN_CH,
    hidden_dim=64,
    out_channels=1,
    n_layers=3,
    dropout=0.3,
).to(DEVICE)

gcn_opt  = torch.optim.Adam(gcn.parameters(), lr=3e-3, weight_decay=1e-4)
gcn_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(gcn_opt, patience=8, factor=0.5)

total_params = sum(p.numel() for p in gcn.parameters())
print(f"GCN  |  params: {total_params:,}  |  in_channels: {IN_CH}")
print()

gcn_history = run_training(gcn, gcn_opt, EPOCHS, use_edge_attr=False, scheduler=gcn_sched)
plot_training(gcn_history, "FootballGCN — Team Prediction")

In [ ]:
gcn_test_loss, gcn_test_acc, gcn_test_auc, gcn_probs, gcn_labels = \
    evaluate(gcn, test_loader, use_edge_attr=False)

print(f"GCN Test  |  loss={gcn_test_loss:.4f}  acc={gcn_test_acc:.3f}  AUC={gcn_test_auc:.3f}")

cm = confusion_matrix(gcn_labels, [int(p > 0.5) for p in gcn_probs])
fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay(cm, display_labels=["Home", "Away"]).plot(ax=ax, colorbar=False)
ax.set_title("GCN — Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 5. GAT Experiment

In [ ]:
gat = FootballGAT(
    in_channels=IN_CH,
    edge_dim=EDGE_DIM,
    hidden_dim=32,
    out_channels=1,
    n_layers=3,
    heads=4,
    dropout=0.3,
).to(DEVICE)

gat_opt   = torch.optim.Adam(gat.parameters(), lr=3e-3, weight_decay=1e-4)
gat_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(gat_opt, patience=8, factor=0.5)

total_params = sum(p.numel() for p in gat.parameters())
print(f"GAT  |  params: {total_params:,}  |  in_channels: {IN_CH}  edge_dim: {EDGE_DIM}")
print()

gat_history = run_training(gat, gat_opt, EPOCHS, use_edge_attr=True, scheduler=gat_sched)
plot_training(gat_history, "FootballGAT — Team Prediction (with edge features)")

In [ ]:
gat_test_loss, gat_test_acc, gat_test_auc, gat_probs, gat_labels = \
    evaluate(gat, test_loader, use_edge_attr=True)

print(f"GAT Test  |  loss={gat_test_loss:.4f}  acc={gat_test_acc:.3f}  AUC={gat_test_auc:.3f}")

cm = confusion_matrix(gat_labels, [int(p > 0.5) for p in gat_probs])
fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay(cm, display_labels=["Home", "Away"]).plot(ax=ax, colorbar=False)
ax.set_title("GAT — Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 6. Head-to-Head Comparison

In [ ]:
from sklearn.metrics import roc_curve

# ── Summary table
results = pd.DataFrame([
    {"Model": "GCN", "Test Acc": gcn_test_acc, "Test AUC": gcn_test_auc, "Test Loss": gcn_test_loss,
     "Params": sum(p.numel() for p in gcn.parameters())},
    {"Model": "GAT", "Test Acc": gat_test_acc, "Test AUC": gat_test_auc, "Test Loss": gat_test_loss,
     "Params": sum(p.numel() for p in gat.parameters())},
])
print(results.to_string(index=False, float_format="{:.4f}".format))
print()

# ── ROC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for (probs, labels, name, color) in [
    (gcn_probs, gcn_labels, "GCN", "#1565c0"),
    (gat_probs, gat_labels, "GAT", "#2e7d32"),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves — Test Set")
axes[0].legend()

# ── Val AUC learning curves
epochs = range(1, EPOCHS + 1)
axes[1].plot(epochs, gcn_history["val_auc"], color="#1565c0", lw=2, label="GCN")
axes[1].plot(epochs, gat_history["val_auc"], color="#2e7d32", lw=2, label="GAT")
axes[1].set_ylim(0.4, 1.0)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val AUC")
axes[1].set_title("Validation AUC over Training")
axes[1].legend()

plt.tight_layout()
plt.show()

winner = results.loc[results["Test AUC"].idxmax(), "Model"]
print(f"\nBest model on test AUC: {winner}")

## 7. Save Best Model

In [ ]:
save_dir = os.path.join(REPO_ROOT, "data", "processed")
os.makedirs(save_dir, exist_ok=True)

torch.save({
    "model_state": gcn.state_dict(),
    "in_channels": IN_CH, "hidden_dim": 64, "out_channels": 1,
    "test_acc": gcn_test_acc, "test_auc": gcn_test_auc,
}, os.path.join(save_dir, "gcn_team_classifier.pt"))

torch.save({
    "model_state": gat.state_dict(),
    "in_channels": IN_CH, "edge_dim": EDGE_DIM, "hidden_dim": 32, "out_channels": 1,
    "test_acc": gat_test_acc, "test_auc": gat_test_auc,
}, os.path.join(save_dir, "gat_team_classifier.pt"))

print(f"Models saved to {save_dir}/")